# 像素级碳密度气候修正
逐年计算生物量/土壤修正系数，并生成修正后地上、地下、土壤及总碳密度栅格。

In [6]:
import numpy as np
import rasterio
from pathlib import Path

In [12]:
# ============================================================
# 路径配置
# ============================================================
LC_DIR  = Path(r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC_from_1")
PRE_DIR = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Precipitation")
TMP_DIR = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Temperature")
KB_DIR  = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\K_B")
KS_DIR  = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\K_S")
OUT_DIR = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Carbon_Density_correction")

for d in [KB_DIR, KS_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

YEARS     = list(range(2023, 2024))
BASE_YEAR = 1990

In [8]:
# ============================================================
# 基准年碳密度表（t/hm²），index = lc_code（1~6）
# 1=Water  2=Forest  3=Grass  4=Bare  5=Impervious  6=Cropland
# ============================================================
C_ABOVE = np.array([0,  0.11, 30.31,  0.90, 13.92, 11.29,  2.59], dtype=np.float32)
C_BELOW = np.array([0,  0.55,  8.23,  3.87,  2.78,  2.26,  0.52], dtype=np.float32)
C_SOIL  = np.array([0,  1.21, 98.30, 12.48,  5.33, 17.97, 45.27], dtype=np.float32)
C_DEAD  = np.array([0,  0.50,  1.36,  3.62,  0.00,  0.00, 13.00], dtype=np.float32)

In [9]:
# ============================================================
# 气候经验模型（Miami + Raich & Schlesinger）
# ============================================================
def npp_p(P):   return 3000 * (1 - np.exp(-0.000664 * P))
def npp_t(T):   return 3000 / (1 + np.exp(1.315 - 0.119 * T))
def soc(P, T):  return 3.996 * np.exp(0.064 * P / (T + 273.15))

# ============================================================
# 读写栅格（LZW 压缩，保留原始地理信息，减小文件体积）
# ============================================================
def read_r(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
        return arr, src.profile

def save_r(arr, profile, path):
    p = profile.copy()
    p.update(dtype="float32", count=1, nodata=-9999,
             compress="lzw", predictor=2,
             tiled=True, blockxsize=256, blockysize=256,
             bigtiff="IF_SAFER")
    out = np.where(np.isnan(arr), -9999, arr).astype(np.float32)
    with rasterio.open(path, "w", **p) as dst:
        dst.write(out, 1)

In [10]:
# ============================================================
# 基准年参考气候值
# ============================================================
P0, _ = read_r(PRE_DIR / f"{BASE_YEAR}.tif")
T0, _ = read_r(TMP_DIR / f"tmp_{BASE_YEAR}_MAT_C.tif")

NPP_P0 = npp_p(P0)
NPP_T0 = npp_t(T0)
SOC_0  = soc(P0, T0)

print(f"基准年 {BASE_YEAR} 参考值计算完成，栅格尺寸: {P0.shape}")

基准年 1990 参考值计算完成，栅格尺寸: (20368, 20289)


In [13]:
# ============================================================
# 主循环：逐年修正
# ============================================================
for year in YEARS:
    lc,  prof = read_r(LC_DIR  / f"LC_from_1_{year}.tif")
    P_t, _    = read_r(PRE_DIR / f"{year}.tif")
    T_t, _    = read_r(TMP_DIR / f"tmp_{year}_MAT_C.tif")
    lc = lc.astype(np.int16)

    # 修正系数
    k_bio  = (npp_p(P_t) / NPP_P0 + npp_t(T_t) / NPP_T0) / 2
    k_soil = soc(P_t, T_t) / SOC_0

    save_r(k_bio,  prof, KB_DIR / f"K_B_{year}.tif")
    save_r(k_soil, prof, KS_DIR / f"K_S_{year}.tif")

    # 查表 + 修正（向量化，只对 lc 1~6 的有效像元操作）
    valid = (lc >= 1) & (lc <= 6)

    cab = np.full(lc.shape, np.nan, dtype=np.float32)
    cbe = np.full(lc.shape, np.nan, dtype=np.float32)
    cso = np.full(lc.shape, np.nan, dtype=np.float32)
    cde = np.full(lc.shape, np.nan, dtype=np.float32)

    lc_v = lc[valid]
    cab[valid] = C_ABOVE[lc_v] * k_bio[valid]
    cbe[valid] = C_BELOW[lc_v] * k_bio[valid]
    cso[valid] = C_SOIL [lc_v] * k_soil[valid]
    cde[valid] = C_DEAD [lc_v]

    # lc=7（原始nodata）及气候数据缺失处一并置 NaN
    mask = ~valid | np.isnan(P_t) | np.isnan(T_t)
    for a in [cab, cbe, cso, cde]:
        a[mask] = np.nan

    save_r(cab,             prof, OUT_DIR / f"C_above_{year}.tif")
    save_r(cbe,             prof, OUT_DIR / f"C_below_{year}.tif")
    save_r(cso,             prof, OUT_DIR / f"C_soil_{year}.tif")
    save_r(cab+cbe+cso+cde, prof, OUT_DIR / f"C_total_{year}.tif")

    print(f"{year}  k_bio={np.nanmean(k_bio):.4f}  k_soil={np.nanmean(k_soil):.4f}  "
          f"C_total均值={np.nanmean(cab+cbe+cso+cde):.2f} t/hm²")

print("\n全部完成。")

2023  k_bio=1.0028  k_soil=0.9832  C_total均值=92.02 t/hm²

全部完成。


In [16]:
############ 计算碳储量 ，CS = C_total * PIXEL_AREA * 1e-4  # t/hm² × 900 m² × (1 hm²/10000 m²) = t/pixel ############
import numpy as np
import rasterio
from pathlib import Path

CTOTAL_DIR = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Carbon_Density_correction")
CS_DIR     = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Carbon_Storage_correction")
CS_DIR.mkdir(parents=True, exist_ok=True)

YEARS      = list(range(2019, 2020))
PIXEL_AREA = 900   # m²，30m×30m

for year in YEARS:
    with rasterio.open(CTOTAL_DIR / f"C_total_{year}.tif") as src:
        arr  = src.read(1).astype(np.float32)
        prof = src.profile
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

    # t/hm² × 900m² × (1hm²/10000m²) = t/pixel
    cs = arr * PIXEL_AREA * 1e-4

    prof.update(dtype="float32", nodata=-9999,
                compress="lzw", predictor=2,
                tiled=True, blockxsize=256, blockysize=256)
    out = np.where(np.isnan(cs), -9999, cs).astype(np.float32)
    with rasterio.open(CS_DIR / f"CS_{year}.tif", "w", **prof) as dst:
        dst.write(out, 1)

    print(f"{year}  总碳储量 = {np.nansum(cs)/1e6:.4f} Tg")

print("\n全部完成。")

2019  总碳储量 = 2216.4611 Tg

全部完成。


In [2]:
# 仅保存枯死物的碳密度栅格到指定文件夹（按年命名）
import numpy as np
import rasterio
from pathlib import Path

# ==============================
# 路径与参数
# ==============================
DEAD_STATIC = {
    1: 0.50,   # Water
    2: 1.36,   # Forest
    3: 3.62,   # Grass
    4: 0.00,   # Bare
    5: 0.00,   # Impervious
    6: 13.00,  # Cropland
}
YEARS = list(range(1990, 2024))
OUT_DIR = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Carbon_Density_correction")
LULC_DIR = Path(r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC_from_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

for year in YEARS:
    lulc_path = LULC_DIR / f"LC_from_1_{year}.tif"
    with rasterio.open(lulc_path) as src:
        lc = src.read(1).astype(np.float32)
        prof = src.profile.copy()
        # 七类设为nan
        lc[lc == 7] = np.nan

    # 构建枯死物碳密度图
    cd_dead = np.full_like(lc, np.nan)
    for code, val in DEAD_STATIC.items():
        cd_dead[lc == code] = val

    # 输出（按年度命名C_dead_{year}.tif，float32，nodata=-9999）
    out_prof = prof.copy()
    out_prof.update(
        dtype="float32",
        nodata=-9999,
        compress="lzw",
        predictor=2,
        tiled=True,
        blockxsize=256,
        blockysize=256,
    )
    out_arr = np.where(np.isnan(cd_dead), -9999, cd_dead).astype(np.float32)

    out_path = OUT_DIR / f"C_dead_{year}.tif"
    with rasterio.open(out_path, "w", **out_prof) as dst:
        dst.write(out_arr, 1)

print("各年份枯死物碳密度tif保存完成。")

各年份枯死物碳密度tif保存完成。


In [3]:
####### 2030年碳计算 #######
# -*- coding: utf-8 -*-
import os
import csv
import numpy as np
import rasterio
from pathlib import Path

# ============================================================
# 1. 输入路径
# ============================================================
LC_2030_PATH = Path(r"G:\Result_30m\Predicted\LC_2030_sim_from_2020.tif")
KB_2020_PATH = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\K_B\K_B_2020.tif")
KS_2020_PATH = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\K_S\K_S_2020.tif")

# ============================================================
# 2. 输出路径
# ============================================================
OUT_DENSITY_DIR = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Carbon_Density_2030_from_simLC")
OUT_STORAGE_DIR = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Carbon_Storage_2030_from_simLC")
OUT_SUMMARY_CSV = Path(r"H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Carbon_Storage_2030_from_simLC\carbon_2030_summary.csv")

OUT_DENSITY_DIR.mkdir(parents=True, exist_ok=True)
OUT_STORAGE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 3. 基准碳密度表（1990静态值，单位：t/hm²）
#    index = lc_code（1~6）
#    1=Water  2=Forest  3=Grass  4=Bare  5=Impervious  6=Cropland
#    index 0 仅作占位
# ============================================================
C_ABOVE = np.array([0,  0.11, 30.31,  0.90, 13.92, 11.29,  2.59], dtype=np.float32)
C_BELOW = np.array([0,  0.55,  8.23,  3.87,  2.78,  2.26,  0.52], dtype=np.float32)
C_SOIL  = np.array([0,  1.21, 98.30, 12.48,  5.33, 17.97, 45.27], dtype=np.float32)
C_DEAD  = np.array([0,  0.50,  1.36,  3.62,  0.00,  0.00, 13.00], dtype=np.float32)

LANDCOVER_NAMES = {
    1: "Water",
    2: "Forest",
    3: "Grass",
    4: "Bare",
    5: "Impervious",
    6: "Cropland",
}

NODATA_OUT = -9999.0

# ============================================================
# 4. 工具函数
# ============================================================
def get_valid_mask(arr, nodata):
    """根据nodata和有限值生成有效掩膜"""
    mask = np.isfinite(arr)
    if nodata is not None:
        mask &= (arr != nodata)
    return mask

def make_out_profile(src_profile):
    """输出栅格配置"""
    prof = src_profile.copy()
    prof.update(
        dtype="float32",
        count=1,
        nodata=NODATA_OUT,
        compress="lzw",
        predictor=2,
        tiled=True,
        bigtiff="IF_SAFER"
    )
    # 若原profile已有block size就保留；否则这里不强行指定
    return prof

def write_array(dst, arr, window):
    out = np.where(np.isfinite(arr), arr, NODATA_OUT).astype(np.float32)
    dst.write(out, 1, window=window)

# ============================================================
# 5. 主程序
# ============================================================
with rasterio.open(LC_2030_PATH) as src_lc, \
     rasterio.open(KB_2020_PATH) as src_kb, \
     rasterio.open(KS_2020_PATH) as src_ks:

    # ----------------------------
    # 一致性检查
    # ----------------------------
    assert src_lc.width == src_kb.width == src_ks.width, "宽度不一致，请先对齐栅格"
    assert src_lc.height == src_kb.height == src_ks.height, "高度不一致，请先对齐栅格"
    assert src_lc.transform == src_kb.transform == src_ks.transform, "transform不一致，请先重采样对齐"
    assert src_lc.crs == src_kb.crs == src_ks.crs, "CRS不一致，请先统一投影"

    out_prof = make_out_profile(src_lc.profile)

    # 像元面积（m²）
    pixel_area_m2 = 900
    # t/hm² -> t/pixel
    area_factor = pixel_area_m2 / 10000.0

    print(f"像元面积: {pixel_area_m2:.4f} m²")
    print(f"面积换算系数（hm²/像元）: {area_factor:.6f}")

    # ----------------------------
    # 输出文件
    # ----------------------------
    density_files = {
        "above": OUT_DENSITY_DIR / "C_above_2030.tif",
        "below": OUT_DENSITY_DIR / "C_below_2030.tif",
        "soil":  OUT_DENSITY_DIR / "C_soil_2030.tif",
        "dead":  OUT_DENSITY_DIR / "C_dead_2030.tif",
        "total": OUT_DENSITY_DIR / "C_total_2030.tif",
    }

    storage_files = {
        "above": OUT_STORAGE_DIR / "CS_above_2030.tif",
        "below": OUT_STORAGE_DIR / "CS_below_2030.tif",
        "soil":  OUT_STORAGE_DIR / "CS_soil_2030.tif",
        "dead":  OUT_STORAGE_DIR / "CS_dead_2030.tif",
        "total": OUT_STORAGE_DIR / "CS_total_2030.tif",
    }

    # 打开输出
    with rasterio.open(density_files["above"], "w", **out_prof) as dst_cab, \
         rasterio.open(density_files["below"], "w", **out_prof) as dst_cbe, \
         rasterio.open(density_files["soil"],  "w", **out_prof) as dst_cso, \
         rasterio.open(density_files["dead"],  "w", **out_prof) as dst_cde, \
         rasterio.open(density_files["total"], "w", **out_prof) as dst_cto, \
         rasterio.open(storage_files["above"], "w", **out_prof) as dst_sab, \
         rasterio.open(storage_files["below"], "w", **out_prof) as dst_sbe, \
         rasterio.open(storage_files["soil"],  "w", **out_prof) as dst_sso, \
         rasterio.open(storage_files["dead"],  "w", **out_prof) as dst_sde, \
         rasterio.open(storage_files["total"], "w", **out_prof) as dst_sto:

        # ----------------------------
        # 汇总量
        # ----------------------------
        sum_storage = {
            "above": 0.0,
            "below": 0.0,
            "soil":  0.0,
            "dead":  0.0,
            "total": 0.0,
        }

        # 也顺带统计各地类面积（km²）和总碳储量（Tg）
        class_pixel_count = {k: 0 for k in range(1, 7)}

        # ----------------------------
        # 分块处理
        # ----------------------------
        for ji, window in src_lc.block_windows(1):
            lc = src_lc.read(1, window=window).astype(np.int16)
            kb = src_kb.read(1, window=window).astype(np.float32)
            ks = src_ks.read(1, window=window).astype(np.float32)

            lc_valid = get_valid_mask(lc.astype(np.float32), src_lc.nodata)
            kb_valid = get_valid_mask(kb, src_kb.nodata)
            ks_valid = get_valid_mask(ks, src_ks.nodata)

            # 有效土地覆盖编码 1~6
            valid_lc_code = (lc >= 1) & (lc <= 6)
            valid = lc_valid & kb_valid & ks_valid & valid_lc_code

            # 初始化
            cab = np.full(lc.shape, np.nan, dtype=np.float32)
            cbe = np.full(lc.shape, np.nan, dtype=np.float32)
            cso = np.full(lc.shape, np.nan, dtype=np.float32)
            cde = np.full(lc.shape, np.nan, dtype=np.float32)
            cto = np.full(lc.shape, np.nan, dtype=np.float32)

            if np.any(valid):
                lc_v = lc[valid]

                # 2030静态碳密度 = 按2030土地覆盖类型查表
                # 2030修正碳密度 = 静态碳密度 × 2020修正因子
                cab[valid] = C_ABOVE[lc_v] * kb[valid]     # biomass
                cbe[valid] = C_BELOW[lc_v] * kb[valid]     # biomass
                cso[valid] = C_SOIL[lc_v]  * ks[valid]     # soil
                cde[valid] = C_DEAD[lc_v]                  # dead, K=1
                cto[valid] = cab[valid] + cbe[valid] + cso[valid] + cde[valid]

                # 各类像元数
                for code in range(1, 7):
                    class_pixel_count[code] += np.sum(lc_v == code)

            # 碳储量（t/pixel）
            sab = cab * area_factor
            sbe = cbe * area_factor
            sso = cso * area_factor
            sde = cde * area_factor
            sto = cto * area_factor

            # 写出碳密度
            write_array(dst_cab, cab, window)
            write_array(dst_cbe, cbe, window)
            write_array(dst_cso, cso, window)
            write_array(dst_cde, cde, window)
            write_array(dst_cto, cto, window)

            # 写出碳储量
            write_array(dst_sab, sab, window)
            write_array(dst_sbe, sbe, window)
            write_array(dst_sso, sso, window)
            write_array(dst_sde, sde, window)
            write_array(dst_sto, sto, window)

            # 汇总总碳储量（t）
            sum_storage["above"] += np.nansum(sab)
            sum_storage["below"] += np.nansum(sbe)
            sum_storage["soil"]  += np.nansum(sso)
            sum_storage["dead"]  += np.nansum(sde)
            sum_storage["total"] += np.nansum(sto)

    # ============================================================
    # 6. 输出汇总表
    # ============================================================
    pixel_area_km2 = pixel_area_m2 / 1e6

    with open(OUT_SUMMARY_CSV, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["Item", "Value", "Unit"])

        # 各类面积
        writer.writerow(["--- Land-cover area summary ---", "", ""])
        for code in range(1, 7):
            area_km2 = class_pixel_count[code] * pixel_area_km2
            writer.writerow([LANDCOVER_NAMES[code], f"{area_km2:.6f}", "km2"])

        # 各碳库总碳储量
        writer.writerow(["--- Carbon storage summary ---", "", ""])
        writer.writerow(["Aboveground carbon storage", f"{sum_storage['above'] / 1e6:.6f}", "Tg"])
        writer.writerow(["Belowground carbon storage", f"{sum_storage['below'] / 1e6:.6f}", "Tg"])
        writer.writerow(["Soil carbon storage",        f"{sum_storage['soil']  / 1e6:.6f}", "Tg"])
        writer.writerow(["Dead organic carbon storage",f"{sum_storage['dead']  / 1e6:.6f}", "Tg"])
        writer.writerow(["Total carbon storage",       f"{sum_storage['total'] / 1e6:.6f}", "Tg"])

    print("\n===== 2030 carbon estimation finished =====")
    print(f"Aboveground carbon storage : {sum_storage['above'] / 1e6:.6f} Tg")
    print(f"Belowground carbon storage : {sum_storage['below'] / 1e6:.6f} Tg")
    print(f"Soil carbon storage        : {sum_storage['soil']  / 1e6:.6f} Tg")
    print(f"Dead organic carbon storage: {sum_storage['dead']  / 1e6:.6f} Tg")
    print(f"Total carbon storage       : {sum_storage['total'] / 1e6:.6f} Tg")
    print(f"Summary CSV saved to: {OUT_SUMMARY_CSV}")

像元面积: 900.0000 m²
面积换算系数（hm²/像元）: 0.090000

===== 2030 carbon estimation finished =====
Aboveground carbon storage : 458.786110 Tg
Belowground carbon storage : 144.423910 Tg
Soil carbon storage        : 1608.955818 Tg
Dead organic carbon storage: 75.458404 Tg
Total carbon storage       : 2287.624250 Tg
Summary CSV saved to: H:\7.Eco_parameter\Hunan_LULC\Revised_Carbon\Carbon_Storage_2030_from_simLC\carbon_2030_summary.csv
